In [4]:
import uuid
import pandas as pd
import os
from datetime import datetime


# ---------------- PLAYER ----------------
class Player:
    def __init__(self, name):
        self.name = name
        self.runs = 0
        self.balls = 0
        self.wickets = 0
        self.runs_conceded = 0


# ---------------- TEAM ----------------
class Team:
    def __init__(self, name):
        self.name = name
        self.players = []
        self.score = 0
        self.wickets = 0


# ---------------- SAVE FUNCTIONS ----------------
def save_all(match):
    os.makedirs("artifacts", exist_ok=True)

    # Ball log
    pd.DataFrame(match.ball_log).to_csv(
        "artifacts/match_log.csv", mode='a', index=False, header=False
    )

    # Batting
    bat = []
    for t in [match.teamA, match.teamB]:
        for p in t.players:
            if p.balls > 0:
                bat.append({
                    "match_id": match.match_id,
                    "team": t.name,
                    "player": p.name,
                    "runs": p.runs,
                    "balls": p.balls
                })
    pd.DataFrame(bat).to_csv("artifacts/batting_scorecard.csv", mode='a', index=False, header=False)

    # Bowling
    bowl = []
    for t in [match.teamA, match.teamB]:
        for p in t.players:
            if p.runs_conceded > 0 or p.wickets > 0:
                bowl.append({
                    "match_id": match.match_id,
                    "team": t.name,
                    "player": p.name,
                    "runs_conceded": p.runs_conceded,
                    "wickets": p.wickets
                })
    pd.DataFrame(bowl).to_csv("artifacts/bowling_scorecard.csv", mode='a', index=False, header=False)

    # Summary CSV
    pd.DataFrame([{
        "match_id": match.match_id,
        "date": datetime.now(),
        "teamA": match.teamA.name,
        "teamA_score": f"{match.teamA.score}/{match.teamA.wickets}",
        "teamB": match.teamB.name,
        "teamB_score": f"{match.teamB.score}/{match.teamB.wickets}",
        "winner": match.winner
    }]).to_csv("artifacts/match_summary.csv", mode='a', index=False, header=False)

    print("✅ ALL DATA SAVED")


# ---------------- MATCH SUMMARY DISPLAY ----------------
def show_match_summary(match):

    print("\n" + "="*40)
    print("🏏 MATCH SUMMARY")
    print("="*40)

    print(f"\n{match.teamA.name}: {match.teamA.score}/{match.teamA.wickets}")
    print(f"{match.teamB.name}: {match.teamB.score}/{match.teamB.wickets}")

    print(f"\n🏆 Winner: {match.winner}")

    all_players = match.teamA.players + match.teamB.players

    # Top batsman
    top_batsman = max(all_players, key=lambda x: x.runs)
    print(f"\n🔥 Top Batsman: {top_batsman.name} ({top_batsman.runs} runs)")

    # Top bowler
    top_bowler = max(all_players, key=lambda x: x.wickets)
    print(f"🎯 Top Bowler: {top_bowler.name} ({top_bowler.wickets} wickets)")

    print("="*40)


# ---------------- MATCH ----------------
class Match:
    def __init__(self, teamA, teamB, overs):
        self.teamA = teamA
        self.teamB = teamB
        self.overs = overs
        self.ball_log = []
        self.match_id = str(uuid.uuid4())[:8]

    def get_input(self):
        valid = ["0","1","2","3","4","6","W","WD","NB"]
        while True:
            b = input("Ball (0,1,2,3,4,6,W,WD,NB): ").upper()
            if b in valid:
                return b
            print("❌ Invalid!")

    def play(self, batting, bowling, innings):

        print(f"\n🏏 Innings {innings}: {batting.name}")

        striker = Player(input("Enter Striker name: "))
        non_striker = Player(input("Enter Non-Striker name: "))
        batting.players.extend([striker, non_striker])

        balls = 0
        bowler = None

        while balls < self.overs * 6 and batting.wickets < 10:

            if balls % 6 == 0:
                bowler = Player(input("\nEnter Bowler name: "))
                bowling.players.append(bowler)

            over = balls//6 + 1
            ball_no = balls%6 + 1

            print(f"\nOver {over}.{ball_no}")
            print(f"{batting.score}/{batting.wickets}")

            ball = self.get_input()

            # LOG
            self.ball_log.append({
                "match_id": self.match_id,
                "innings": innings,
                "over": over,
                "ball": ball_no,
                "batsman": striker.name,
                "bowler": bowler.name,
                "event": ball
            })

            if ball == "W":
                batting.wickets += 1
                bowler.wickets += 1
                print(f"{striker.name} OUT!")

                striker = Player(input("Enter next batsman: "))
                batting.players.append(striker)

            elif ball in ["WD","NB"]:
                batting.score += 1
                continue

            else:
                runs = int(ball)
                batting.score += runs
                striker.runs += runs
                striker.balls += 1
                bowler.runs_conceded += runs

                if runs % 2 == 1:
                    striker, non_striker = non_striker, striker

            balls += 1

            if balls % 6 == 0:
                striker, non_striker = non_striker, striker

        return batting.score

    def start(self):
        print(f"\n🆔 Match ID: {self.match_id}")

        target = self.play(self.teamA, self.teamB, 1) + 1
        print(f"\n🎯 Target: {target}")

        score2 = self.play(self.teamB, self.teamA, 2)

        if score2 >= target:
            print(f"\n🎉 {self.teamB.name} WON!")
            self.winner = self.teamB.name
        else:
            print(f"\n🏆 {self.teamA.name} WON!")
            self.winner = self.teamA.name

        # SAVE
        save_all(self)

        # SHOW SUMMARY 🔥
        show_match_summary(self)


# ---------------- RUN ----------------
os.makedirs("artifacts", exist_ok=True)

teamA_name = input("Enter Team A: ")
teamB_name = input("Enter Team B: ")

overs = int(input("Enter overs (2/20/50): "))

teamA = Team(teamA_name)
teamB = Team(teamB_name)

match = Match(teamA, teamB, overs)
match.start()

Enter Team A:  w
Enter Team B:  w
Enter overs (2/20/50):  2



🆔 Match ID: a9aa4721

🏏 Innings 1: w


Enter Striker name:  visal
Enter Non-Striker name:  ravindu

Enter Bowler name:  linuka



Over 1.1
0/0


Ball (0,1,2,3,4,6,W,WD,NB):  1


...


Over 2.5
15/1


Ball (0,1,2,3,4,6,W,WD,NB):  2



Over 2.6
17/1


Ball (0,1,2,3,4,6,W,WD,NB):  2



🏆 w WON!
✅ ALL DATA SAVED

🏏 MATCH SUMMARY

w: 22/0
w: 19/1

🏆 Winner: w

🔥 Top Batsman: ravindu (17 runs)
🎯 Top Bowler: 1 (1 wickets)


In [2]:
def show_match_summary(match):

    print("\n" + "="*40)
    print("🏏 MATCH SUMMARY")
    print("="*40)

    print(f"\n{match.teamA.name}: {match.teamA.score}/{match.teamA.wickets}")
    print(f"{match.teamB.name}: {match.teamB.score}/{match.teamB.wickets}")

    print(f"\n🏆 Winner: {match.winner}")

    # -------- TOP BATSMAN --------
    all_players = match.teamA.players + match.teamB.players

    top_batsman = max(all_players, key=lambda x: x.runs)
    print(f"\n🔥 Top Batsman: {top_batsman.name} ({top_batsman.runs} runs)")

    # -------- TOP BOWLER --------
    top_bowler = max(all_players, key=lambda x: x.wickets)
    print(f"🎯 Top Bowler: {top_bowler.name} ({top_bowler.wickets} wickets)")

    print("="*40)

In [5]:
show_match_summary(match)


🏏 MATCH SUMMARY

w: 22/0
w: 19/1

🏆 Winner: w

🔥 Top Batsman: ravindu (17 runs)
🎯 Top Bowler: 1 (1 wickets)
